# WEEK 6 — FEATURE ENGINEERING
## Road Accident Dataset
### Normalization • Scaling • Categorical Encoding • Fixing Skewness

**Goal:** prepare features from the cleaned road-accident dataset and visualize the data using **20 colorful graphs**.

**Dataset:** the existing cleaned Week 3 road-accident CSV. This notebook tries common filenames automatically, so keep the CSV in the same folder as this notebook.

**Important:** original columns are preserved. New features and transformed representations are created separately. Numeric codes such as road type or accident severity are treated as categories where appropriate. Graphs use black, white, and grayscale only.

## 1. Import Libraries

This cell imports the libraries for data handling, feature transformation, statistical summaries, and visualization.

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#17becf", "#e377c2", "#bcbd22"])
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["text.color"] = "black"
plt.rcParams["axes.labelcolor"] = "black"
plt.rcParams["xtick.color"] = "black"
plt.rcParams["ytick.color"] = "black"

### Expected result

Libraries load and all charts use grayscale. If a library is missing, install it in your notebook environment with `pip install pandas numpy matplotlib scipy scikit-learn`.

## 2. Locate and Load the Cleaned Dataset

The notebook checks several common cleaned dataset filenames in the notebook folder. If none is found, set `DATA_PATH` to the exact CSV filename.

In [ ]:
possible_files = [
    "AccidentsBig_cleaned_week4_FINAL.csv",
    "AccidentsBig_cleaned_week4.csv",
    "AccidentsBig_cleaned_week3.csv",
    "AccidentsBig_cleaned_week3_2.csv",
    "AccidentsBig.csv"
]
DATA_PATH = next((f for f in possible_files if os.path.exists(f)), None)

if DATA_PATH is None:
    csv_files = glob.glob("*.csv")
    if len(csv_files) == 1:
        DATA_PATH = csv_files[0]
    else:
        raise FileNotFoundError(
            "Place the cleaned road-accident CSV in the same folder as this notebook. "
            "Expected one of: " + ", ".join(possible_files)
        )

df = pd.read_csv(DATA_PATH, low_memory=False)
print("Loaded file:", DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())

### What to check

Confirm that the correct cleaned road-accident CSV was loaded and that the preview shows accident details such as severity, vehicles, casualties, date, time, and road/environment variables.

## 3. Check Dataset Quality

This cell checks dimensions, missing values, exact duplicate rows, duplicate accident identifiers, and column types. It reports issues rather than silently removing accident records.

In [ ]:
print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Missing values:", int(df.isna().sum().sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))
if "Accident_Index" in df.columns:
    print("Duplicate Accident_Index values:", int(df["Accident_Index"].duplicated().sum()))
display(pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum(),
    "Unique Values": df.nunique(dropna=False)
}))

### Interpretation

Use this output to document the state of the input file. Duplicate IDs are flagged for review because an ID should identify an accident, but records should not be deleted without confirming the dataset's structure.

## 4. Create a Working Copy and Date/Time Features

The original `df` remains unchanged. The working copy derives date parts and hour/time period when those columns are available.

In [ ]:
feature_df = df.copy()

if "Date" in feature_df.columns:
    parsed_date = pd.to_datetime(feature_df["Date"], dayfirst=True, errors="coerce")
    feature_df["Accident_Year"] = parsed_date.dt.year
    feature_df["Accident_Month"] = parsed_date.dt.month
    feature_df["Accident_Day"] = parsed_date.dt.day
    feature_df["Accident_Weekday_From_Date"] = parsed_date.dt.dayofweek

if "Time" in feature_df.columns:
    parsed_time = pd.to_datetime(feature_df["Time"].astype("string"), format="%H:%M", errors="coerce")
    feature_df["Accident_Hour"] = parsed_time.dt.hour
    feature_df["Time_Period"] = pd.cut(
        feature_df["Accident_Hour"], [-1, 5, 11, 17, 23],
        labels=["Night", "Morning", "Afternoon", "Evening"]
    )

print("Original shape:", df.shape)
print("Feature-engineered shape:", feature_df.shape)
display(feature_df.head())

### Interpretation

Date/time features can make it easier to compare accidents by month, weekday, hour, or broad time period. Unparseable values remain missing instead of being guessed.

## 5. Prepare Columns for Analysis

Use known road-accident columns when available. Coded categorical fields remain categorical for encoding and category charts.

In [ ]:
def existing(columns):
    return [c for c in columns if c in feature_df.columns]

numeric_candidates = existing([
    "Number_of_Vehicles", "Number_of_Casualties", "Speed_limit",
    "longitude", "latitude", "Accident_Hour", "Accident_Month"
])
category_candidates = existing([
    "Accident_Severity", "Day_of_Week", "Road_Type", "Speed_limit",
    "Light_Conditions", "Weather_Conditions", "Road_Surface_Conditions",
    "Urban_or_Rural_Area", "Junction_Detail", "Junction_Control",
    "Time_Period", "Accident_Month"
])
print("Numeric columns:", numeric_candidates)
print("Categorical/code columns:", category_candidates)

### Interpretation

The notebook adapts to the columns that exist in the CSV. If a field is missing, the relevant chart is skipped rather than stopping the entire notebook.

## Graph 1 — Accident Severity Counts

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if "Accident_Severity" in feature_df:
    s = feature_df["Accident_Severity"].astype("string").fillna("Missing").value_counts().sort_index()
    plt.figure(figsize=(7,4)); s.plot(kind="bar", color="#4C78A8", edgecolor="black")
    plt.title("Accident Severity Distribution"); plt.xlabel("Severity Code"); plt.ylabel("Accident Count")
    plt.tight_layout(); plt.show()
else: print("Skipped: Accident_Severity column unavailable.")

### Interpretation

This bar chart compares the number of records in each accident-severity category. The dataset's original codes are shown because a verified codebook is not provided.

## Graph 2 — Number of Vehicles Histogram

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if "Number_of_Vehicles" in feature_df:
    x = pd.to_numeric(feature_df["Number_of_Vehicles"], errors="coerce").dropna()
    plt.figure(figsize=(7,4)); plt.hist(x, bins=25, color="#4C78A8", edgecolor="black")
    plt.title("Distribution of Number of Vehicles"); plt.xlabel("Number of Vehicles"); plt.ylabel("Frequency")
    plt.tight_layout(); plt.show()
else: print("Skipped: Number_of_Vehicles column unavailable.")

### Interpretation

The histogram shows the distribution and concentration of vehicles involved per accident.

## Graph 3 — Number of Casualties Histogram

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if "Number_of_Casualties" in feature_df:
    x = pd.to_numeric(feature_df["Number_of_Casualties"], errors="coerce").dropna()
    plt.figure(figsize=(7,4)); plt.hist(x, bins=25, color="#4C78A8", edgecolor="black")
    plt.title("Distribution of Number of Casualties"); plt.xlabel("Number of Casualties"); plt.ylabel("Frequency")
    plt.tight_layout(); plt.show()
else: print("Skipped: Number_of_Casualties column unavailable.")

### Interpretation

This graph shows how many casualties are recorded for each accident and can reveal a right-skewed count distribution.

## Graph 4 — Speed Limit Frequency

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if "Speed_limit" in feature_df:
    s = feature_df["Speed_limit"].value_counts().sort_index()
    plt.figure(figsize=(7,4)); s.plot(kind="bar", color="#4C78A8", edgecolor="black")
    plt.title("Accidents by Speed Limit"); plt.xlabel("Speed Limit"); plt.ylabel("Accident Count")
    plt.tight_layout(); plt.show()
else: print("Skipped: Speed_limit column unavailable.")

### Interpretation

This graph compares accident-record frequencies across the speed-limit values present in the dataset.

## Graph 5 — Accidents by Day of Week

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if "Day_of_Week" in feature_df:
    s = feature_df["Day_of_Week"].value_counts().sort_index()
    plt.figure(figsize=(7,4)); s.plot(kind="bar", color="#4C78A8", edgecolor="black")
    plt.title("Accidents by Day of Week"); plt.xlabel("Day-of-Week Code"); plt.ylabel("Accident Count")
    plt.tight_layout(); plt.show()
else: print("Skipped: Day_of_Week column unavailable.")

### Interpretation

The plot compares records by day-of-week code. Interpret the codes using the dataset's documentation if available.

## Graph 6 — Road Type Frequency

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if "Road_Type" in feature_df:
    s = feature_df["Road_Type"].astype("string").fillna("Missing").value_counts().sort_index()
    plt.figure(figsize=(8,4)); s.plot(kind="bar", color="#4C78A8", edgecolor="black")
    plt.title("Accidents by Road Type"); plt.xlabel("Road Type Code"); plt.ylabel("Accident Count")
    plt.tight_layout(); plt.show()
else: print("Skipped: Road_Type column unavailable.")

### Interpretation

This chart compares accident frequencies among road-type codes without assigning unverified labels.

## Graph 7 — Weather Conditions Frequency

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if "Weather_Conditions" in feature_df:
    s = feature_df["Weather_Conditions"].astype("string").fillna("Missing").value_counts().sort_index()
    plt.figure(figsize=(8,4)); s.plot(kind="bar", color="#4C78A8", edgecolor="black")
    plt.title("Accidents by Weather Condition"); plt.xlabel("Weather Code"); plt.ylabel("Accident Count")
    plt.tight_layout(); plt.show()
else: print("Skipped: Weather_Conditions column unavailable.")

### Interpretation

This chart shows how the accident records are distributed across recorded weather-condition codes; it does not establish causation.

## Graph 8 — Light Conditions Frequency

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if "Light_Conditions" in feature_df:
    s = feature_df["Light_Conditions"].astype("string").fillna("Missing").value_counts().sort_index()
    plt.figure(figsize=(8,4)); s.plot(kind="bar", color="#4C78A8", edgecolor="black")
    plt.title("Accidents by Light Condition"); plt.xlabel("Light Code"); plt.ylabel("Accident Count")
    plt.tight_layout(); plt.show()
else: print("Skipped: Light_Conditions column unavailable.")

### Interpretation

This chart compares counts across light-condition codes.

## Graph 9 — Road Surface Conditions Frequency

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if "Road_Surface_Conditions" in feature_df:
    s = feature_df["Road_Surface_Conditions"].astype("string").fillna("Missing").value_counts().sort_index()
    plt.figure(figsize=(8,4)); s.plot(kind="bar", color="#4C78A8", edgecolor="black")
    plt.title("Accidents by Road Surface Condition"); plt.xlabel("Surface Code"); plt.ylabel("Accident Count")
    plt.tight_layout(); plt.show()
else: print("Skipped: Road_Surface_Conditions column unavailable.")

### Interpretation

This graph summarizes the recorded road-surface-condition categories.

## Graph 10 — Urban/Rural Frequency

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if "Urban_or_Rural_Area" in feature_df:
    s = feature_df["Urban_or_Rural_Area"].astype("string").fillna("Missing").value_counts().sort_index()
    plt.figure(figsize=(7,4)); s.plot(kind="bar", color="#4C78A8", edgecolor="black")
    plt.title("Accidents by Urban/Rural Code"); plt.xlabel("Area Code"); plt.ylabel("Accident Count")
    plt.tight_layout(); plt.show()
else: print("Skipped: Urban_or_Rural_Area column unavailable.")

### Interpretation

This chart compares recorded accidents across the urban/rural area codes.

## Graph 11 — Vehicles vs Casualties Scatter Plot

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if {"Number_of_Vehicles", "Number_of_Casualties"}.issubset(feature_df.columns):
    sample = feature_df[["Number_of_Vehicles","Number_of_Casualties"]].apply(pd.to_numeric, errors="coerce").dropna()
    if len(sample) > 10000: sample = sample.sample(10000, random_state=42)
    plt.figure(figsize=(7,4)); plt.scatter(sample["Number_of_Vehicles"], sample["Number_of_Casualties"],
        color="#F58518", edgecolors="black", alpha=0.5, s=18)
    plt.title("Number of Vehicles vs Number of Casualties"); plt.xlabel("Vehicles"); plt.ylabel("Casualties")
    plt.tight_layout(); plt.show()
else: print("Skipped: required columns unavailable.")

### Interpretation

The scatter plot checks whether higher vehicle counts tend to appear alongside higher casualty counts. A visible association is not proof of causation.

## Graph 12 — Casualties by Accident Severity

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if {"Accident_Severity", "Number_of_Casualties"}.issubset(feature_df.columns):
    tmp = feature_df[["Accident_Severity","Number_of_Casualties"]].copy()
    tmp["Number_of_Casualties"] = pd.to_numeric(tmp["Number_of_Casualties"], errors="coerce")
    groups = [g["Number_of_Casualties"].dropna().values for _, g in tmp.groupby("Accident_Severity", dropna=True)]
    labels = [str(k) for k, _ in tmp.groupby("Accident_Severity", dropna=True)]
    if groups:
        plt.figure(figsize=(7,4)); plt.boxplot(groups, labels=labels)
        plt.title("Casualties by Accident Severity"); plt.xlabel("Severity Code"); plt.ylabel("Casualties")
        plt.tight_layout(); plt.show()
else: print("Skipped: required columns unavailable.")

### Interpretation

Boxplots compare the median, spread, and potential extreme values of casualty counts across severity codes.

## Graph 13 — Severity by Weather Condition

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if {"Weather_Conditions","Accident_Severity"}.issubset(feature_df.columns):
    tab = pd.crosstab(feature_df["Weather_Conditions"], feature_df["Accident_Severity"])
    tab.plot(kind="bar", figsize=(8,4), color=["#4C78A8", "#F58518", "#54A24B", "#E45756"])
    plt.title("Accident Severity by Weather Code"); plt.xlabel("Weather Code"); plt.ylabel("Record Count")
    plt.tight_layout(); plt.show()
else: print("Skipped: required columns unavailable.")

### Interpretation

Grouped bars compare severity-code counts within weather categories. Category labels remain numeric codes.

## Graph 14 — Severity by Light Condition

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if {"Light_Conditions","Accident_Severity"}.issubset(feature_df.columns):
    tab = pd.crosstab(feature_df["Light_Conditions"], feature_df["Accident_Severity"])
    tab.plot(kind="bar", figsize=(8,4), color=["#4C78A8", "#F58518", "#54A24B", "#E45756"])
    plt.title("Accident Severity by Light Code"); plt.xlabel("Light Code"); plt.ylabel("Record Count")
    plt.tight_layout(); plt.show()
else: print("Skipped: required columns unavailable.")

### Interpretation

This graph compares the severity distribution across recorded light-condition codes.

## Graph 15 — Severity by Road Surface

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if {"Road_Surface_Conditions","Accident_Severity"}.issubset(feature_df.columns):
    tab = pd.crosstab(feature_df["Road_Surface_Conditions"], feature_df["Accident_Severity"])
    tab.plot(kind="bar", figsize=(8,4), color=["#4C78A8", "#F58518", "#54A24B", "#E45756"])
    plt.title("Accident Severity by Road Surface Code"); plt.xlabel("Surface Code"); plt.ylabel("Record Count")
    plt.tight_layout(); plt.show()
else: print("Skipped: required columns unavailable.")

### Interpretation

This plot compares severity counts by road-surface code and should be interpreted descriptively.

## Graph 16 — Accidents by Hour

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if "Accident_Hour" in feature_df:
    s = feature_df["Accident_Hour"].value_counts().sort_index()
    plt.figure(figsize=(8,4)); plt.plot(s.index, s.values, color="#2CA02C", marker="o", markersize=3)
    plt.title("Accident Records by Hour of Day"); plt.xlabel("Hour (0–23)"); plt.ylabel("Accident Count")
    plt.xticks(range(0,24,2)); plt.grid(True, color="lightgray", linewidth=0.5)
    plt.tight_layout(); plt.show()
else: print("Skipped: Accident_Hour could not be created.")

### Interpretation

The line chart shows the distribution of accident records by hour, based on parseable times.

## Graph 17 — Accidents by Month

The following cell generates this graph from the columns available in the dataset.

In [ ]:
if "Accident_Month" in feature_df:
    s = feature_df["Accident_Month"].value_counts().sort_index()
    plt.figure(figsize=(8,4)); plt.plot(s.index, s.values, color="#2CA02C", marker="o")
    plt.title("Accident Records by Month"); plt.xlabel("Month"); plt.ylabel("Accident Count")
    plt.xticks(range(1,13)); plt.grid(True, color="lightgray", linewidth=0.5)
    plt.tight_layout(); plt.show()
else: print("Skipped: Accident_Month could not be created.")

### Interpretation

This chart shows monthly record counts for the dates successfully parsed from the dataset.

## Graph 18 — Correlation Heatmap

The following cell generates this graph from the columns available in the dataset.

In [ ]:
corr_cols = [c for c in ["Number_of_Vehicles","Number_of_Casualties","Speed_limit","longitude","latitude"]
             if c in feature_df.columns and pd.api.types.is_numeric_dtype(feature_df[c])]
if len(corr_cols) >= 2:
    corr = feature_df[corr_cols].apply(pd.to_numeric, errors="coerce").corr()
    fig, ax = plt.subplots(figsize=(7,5))
    im = ax.imshow(corr, cmap="viridis", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr_cols))); ax.set_xticklabels(corr_cols, rotation=45, ha="right")
    ax.set_yticks(range(len(corr_cols))); ax.set_yticklabels(corr_cols)
    for i in range(len(corr_cols)):
        for j in range(len(corr_cols)):
            if pd.notna(corr.iloc[i,j]): ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", color="black")
    ax.set_title("Correlation Heatmap of Selected Numeric Variables")
    fig.colorbar(im, ax=ax, label="Correlation")
    plt.tight_layout(); plt.show()
else: print("Skipped: insufficient numeric columns for correlation.")

### Interpretation

Correlation indicates linear association among numeric variables. Numeric category codes should not be treated as continuous measurements in this heatmap.

## Graph 19 — Skewness Before vs After Log1p

The following cell generates this graph from the columns available in the dataset.

In [ ]:
skew_candidates = [c for c in ["Number_of_Vehicles","Number_of_Casualties","Speed_limit"]
                   if c in feature_df.columns and pd.api.types.is_numeric_dtype(feature_df[c])]
rows = []
for col in skew_candidates:
    x = pd.to_numeric(feature_df[col], errors="coerce").dropna()
    if len(x) > 2 and x.nunique() > 2 and (x >= 0).all():
        before = skew(x, bias=False)
        after = skew(np.log1p(x), bias=False)
        rows.append((col, before, after))
if rows:
    skew_df = pd.DataFrame(rows, columns=["Feature","Before","After log1p"]).set_index("Feature")
    skew_df.plot(kind="bar", figsize=(8,4), color=["#4C78A8","#F58518"], edgecolor="black")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.title("Skewness Before and After log1p"); plt.ylabel("Skewness")
    plt.xticks(rotation=0); plt.tight_layout(); plt.show()
else: print("Skipped: no suitable non-negative numeric features.")

### Interpretation

The comparison shows whether log1p reduces skewness for eligible variables. It is a diagnostic, not a guarantee that the transformed variable is normally distributed.

## Graph 20 — Min-Max Scaling Comparison

The following cell generates this graph from the columns available in the dataset.

In [ ]:
scale_cols = [c for c in ["Number_of_Vehicles","Number_of_Casualties","Speed_limit"]
              if c in feature_df.columns and pd.api.types.is_numeric_dtype(feature_df[c])]
if scale_cols:
    scale_data = feature_df[scale_cols].apply(pd.to_numeric, errors="coerce")
    scale_data = scale_data.fillna(scale_data.median())
    scaled = MinMaxScaler().fit_transform(scale_data)
    scaled_df = pd.DataFrame(scaled, columns=scale_cols)
    compare = pd.DataFrame({
        "Original mean": scale_data.mean(),
        "Scaled mean": scaled_df.mean(),
        "Original std": scale_data.std(),
        "Scaled std": scaled_df.std()
    })
    compare[["Original mean","Scaled mean"]].plot(kind="bar", figsize=(8,4), color=["#4C78A8","#F58518"], edgecolor="black")
    plt.title("Original vs Min-Max Scaled Means"); plt.ylabel("Mean")
    plt.xticks(rotation=0); plt.tight_layout(); plt.show()
    display(compare.round(4))
else: print("Skipped: no numeric columns available for scaling.")

### Interpretation

Min-Max scaling changes numeric feature ranges to 0–1. The comparison illustrates the change in scale; it does not change the underlying accident records.

## 6. One-Hot Encoding Selected Categorical Variables

Create a separate encoded representation for a manageable set of categories. This output is not merged into the full dataset, preserving original data and keeping memory use more manageable.

In [ ]:
encode_cols = [c for c in ["Road_Type", "Light_Conditions", "Weather_Conditions",
    "Road_Surface_Conditions", "Urban_or_Rural_Area"] if c in feature_df.columns]
if encode_cols:
    enc_input = feature_df[encode_cols].astype("string").fillna("Unknown")
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.int8)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False, dtype=np.int8)
    encoded_df = pd.DataFrame(encoder.fit_transform(enc_input),
        columns=encoder.get_feature_names_out(encode_cols), index=feature_df.index)
    print("Encoded columns:", encode_cols)
    print("Encoded shape:", encoded_df.shape)
    display(encoded_df.head())
else:
    encoded_df = pd.DataFrame(index=feature_df.index)
    print("No selected categorical fields were found.")

### Interpretation

Each new encoded column represents a category from the selected fields. The encoded table is a separate modeling representation; the original category columns remain unchanged.

## 7. Standardization with Z-Scores

Standardize selected numerical features and store them in a separate DataFrame.

In [ ]:
numeric_scale_cols = [c for c in ["Number_of_Vehicles","Number_of_Casualties","Speed_limit","Accident_Hour","Accident_Month"]
                      if c in feature_df.columns and pd.api.types.is_numeric_dtype(feature_df[c])]
if numeric_scale_cols:
    numeric_data = feature_df[numeric_scale_cols].apply(pd.to_numeric, errors="coerce")
    numeric_data = numeric_data.fillna(numeric_data.median())
    standardized_df = pd.DataFrame(StandardScaler().fit_transform(numeric_data),
        columns=[f"{c}_ZScore" for c in numeric_scale_cols], index=feature_df.index)
    display(standardized_df.head())
    display(standardized_df.agg(["mean","std"]).T.round(3))
else:
    standardized_df = pd.DataFrame(index=feature_df.index)
    print("No numeric columns available for standardization.")

### Interpretation

Standardized values are expressed in standard-deviation units. For model training, scalers should be fit on training data only to prevent data leakage.

## 8. Save the Feature-Engineered Dataset (Optional)

The output includes the original fields plus derived date/time features. It does not automatically merge scaled or one-hot columns into the main CSV.

In [ ]:
OUTPUT_PATH = "road_accidents_week6_feature_engineered.csv"
# Uncomment the next line to save the feature-engineered dataset:
# feature_df.to_csv(OUTPUT_PATH, index=False)
print("Rows and columns in feature_df:", feature_df.shape)
print("Rows and columns in encoded_df:", encoded_df.shape)
print("Rows and columns in standardized_df:", standardized_df.shape)

### Final note

Week 6 covered feature engineering, normalization, scaling, categorical encoding, and skewness using 20 colorful graphs. The notebook preserves original values and labels coded categories as codes. When building a predictive model, fit all transformations using the training split only.